In [ ]:
import os
import re
import time
import math
import string
import random
import joblib
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import StratifiedKFold, GroupKFold, KFold
from sklearn.metrics import confusion_matrix, classification_report
from collections import defaultdict
from textwrap import wrap
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential, load_model
from keras.layers import Dense, Embedding, LSTM, Dropout, Flatten, Conv1D, GlobalMaxPooling1D, MaxPooling1D, SpatialDropout1D, GRU, Bidirectional
from keras.initializers import Constant
from keras.callbacks import ModelCheckpoint
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, SGD, AdamW
from torch.utils.data import DataLoader, Dataset
import transformers
from transformers import AutoTokenizer, AutoModel, AutoConfig
from transformers import get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

sns.set(style='whitegrid', palette='muted', font_scale=1.2)
HAPPY_COLORS_PALETTE = ["#01BEFE", "#FFDD00", "#FF7D00", "#FF006D", "#ADFF02", "#8F00FF"]
sns.set_palette(sns.color_palette(HAPPY_COLORS_PALETTE))
plt.rcParams['figure.figsize'] = 12, 8
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

nltk.download('wordnet')
nltk.download('punkt')
nltk.download('stopwords')


In [ ]:
#For colab
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load the dataset
datapath = '/content/drive/MyDrive/HateSpeech_Results/Dataset/Binary_BDHate.xlsx'
df = pd.read_excel(datapath)
df.head()

In [ ]:
#for Binary if you are using kaggle 
import pandas as pd
datapath = '/kaggle/input/binary-bd-hate-speech/Binary_BDHate.xlsx'
df = pd.read_excel(datapath)
df.head()

In [ ]:
#for multiclass, if you are using kaggle
datapath = '/content/drive/MyDrive/HateSpeech_Results/Dataset/multi_HS.xlsx'
df = pd.read_excel(datapath)
df.head()

In [ ]:
import pandas as pd
datapath = '/kaggle/input/multi-hs/multi_HS.xlsx'
df = pd.read_excel(datapath)
df.head()

In [ ]:
# Group by 'Label' and get the size of each group
label_counts = df.groupby(['Label']).size()

# Print the total data for each category
print("Total data for each category:")
print(label_counts)

In [ ]:
# Group by 'Label' and get the size of each group
label_counts = df.groupby(['Category']).size()

# Print the total data for each category
print("Total data for each category:")
print(label_counts)

In [ ]:
def cleaning_data(row):
    text = re.sub('[^\u0980-\u09FF]',' ',str(row)) 
    return text

In [ ]:
import re

def cleaning_data(row):
    
    text = re.sub(r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F700-\U0001F77F\U0001F780-\U0001F7FF\U0001F800-\U0001F8FF\U0001F900-\U0001F9FF\U0001FA00-\U0001FA6F\U0001FA70-\U0001FAFF\U00002702-\U000027B0\U000024C2-\U0001F251\U0001f926-\U0001f937\U0001F1E0-\U0001F1FF]', '', str(row))
    
    text = re.sub(r'[^\w\s\u0980-\u09FF]', '', text)
    return text


In [ ]:
df['Comments'] = df['Comments'].apply(cleaning_data)

In [ ]:
df.head()

# Muticlass Hate Speech Classification 

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

# Define the labels dictionary with a different name
label_dict = {
    'anti-political': 0,
    'anti-religious': 1,
    'misogynist': 2,
    'xenophobic': 4,
    'slander': 3
}

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained('csebuetnlp/banglabert')

class HateSpeechDataset(Dataset):
    def __init__(self, texts, categories, tokenizer, max_len):
        self.texts = texts
        self.categories = categories
        self.tokenizer = tokenizer
        self.max_len = max_len

        # Encode categories manually using the label_dict dictionary
        self.labels = [label_dict[category] for category in categories]
 

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        inputs = self.tokenizer(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_token_type_ids=False,
            return_attention_mask=True,
            return_tensors="pt",
        )

        return {
            "input_ids": inputs["input_ids"].flatten(),
            "attention_mask": inputs["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long),
        }

# Assuming you have a dataframe `df` with 'Comments' and 'Category' columns
texts = df['Comments'].tolist()
categories = df['Category'].tolist()

# Split data into train, validation, and test sets
train_texts, test_texts, train_labels, test_labels = train_test_split(texts, categories, test_size=0.2, random_state=42, stratify=categories)
val_texts, test_texts, val_labels, test_labels = train_test_split(test_texts, test_labels, test_size=0.5, random_state=42, shuffle=False)

# Define training, validation, and test datasets
train_dataset = HateSpeechDataset(train_texts, train_labels, tokenizer, max_len=60)
val_dataset = HateSpeechDataset(val_texts, val_labels, tokenizer, max_len=60)
test_dataset = HateSpeechDataset(test_texts, test_labels, tokenizer, max_len=60)

# Define dataloaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))


In [ ]:

token_lens = []
for txt in df.Comments:
  tokens = tokenizer.encode(txt, max_length=512,truncation=True)
  token_lens.append(len(tokens))

HateBert-LSTM

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np

class BertLSTMClassifier(nn.Module):
    def __init__(self, bert_model, lstm_hidden_size, num_classes):
        super(BertLSTMClassifier, self).__init__()
        self.bert = bert_model
        self.lstm1 = nn.LSTM(input_size=self.bert.config.hidden_size,
                             hidden_size=lstm_hidden_size,
                             num_layers=1,
                             batch_first=True,
                             dropout=0.3,
                             bidirectional=False)
        self.dropout1 = nn.Dropout(0.3)
        self.lstm2 = nn.LSTM(input_size=lstm_hidden_size,
                             hidden_size=lstm_hidden_size,
                             num_layers=1,
                             batch_first=True,
                             dropout=0.3,
                             bidirectional=False)
        self.dropout2 = nn.Dropout(0.3)
        self.global_max_pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(lstm_hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        # BERT encoding
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        bert_output = outputs.last_hidden_state

        # First LSTM layer
        lstm_output1, _ = self.lstm1(bert_output)
        lstm_output1 = self.dropout1(lstm_output1)

        # Second LSTM layer
        lstm_output2, _ = self.lstm2(lstm_output1)
        lstm_output2, _ = self.dropout2(lstm_output2)
        # Global Max Pooling
        lstm_output2 = lstm_output2.permute(0, 2, 1)
        pooled_output = self.global_max_pool(lstm_output2).squeeze(2)

        # Classification layer
        logits = self.fc(pooled_output)

        return logits
def train(model, train_loader, val_loader, optimizer, criterion, scheduler, device, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        total_samples = 0

        progress_bar_train = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs} - Training')
        for batch in progress_bar_train:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            progress_bar_train.set_postfix({'Train Loss': total_loss / total_samples,
                                            'Train Accuracy': total_correct / total_samples})

        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {total_loss / total_samples:.4f}, "
              f"Train Accuracy: {total_correct / total_samples:.4f}, "
              f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

        # Step the scheduler after each epoch
        scheduler.step()

def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    val_loss = total_loss / len(val_loader)
    val_accuracy = total_correct / total_samples

    return val_loss, val_accuracy

def test(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_loss = total_loss / len(test_loader)
    test_accuracy = total_correct / total_samples

    return test_loss, test_accuracy, all_predictions, all_labels

# Initialize the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('csebuetnlp/banglabert')
model = BertLSTMClassifier(AutoModel.from_pretrained('csebuetnlp/banglabert'), lstm_hidden_size=512, num_classes=5)

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define optimizer and criterion
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# Define the learning rate scheduler
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.9)

# Train the model
train(model, train_loader, val_loader, optimizer, criterion, scheduler, device, epochs=15)



HateBert-MLP

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score
from tqdm import tqdm


class BertClassifier(nn.Module):
    def __init__(self, bert_model, num_classes):
        super(BertClassifier, self).__init__()
        self.bert = bert_model
        
        hidden_size = self.bert.config.hidden_size
        
        # Global Max Pooling
        self.global_max_pool = nn.AdaptiveMaxPool1d(1)
        
        # Multi-Layer Perceptron (MLP)
        self.fc1 = nn.Linear(hidden_size, 512)
        self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(512, 256)
        self.dropout2 = nn.Dropout(0.3)
        
        # Final classification layer
        self.fc_out = nn.Linear(256, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state  # shape: (batch_size, seq_len, hidden_size)

       
        pooled = self.global_max_pool(last_hidden_state.permute(0, 2, 1)).squeeze(-1)  # (batch_size, hidden_size)
    
        x = torch.relu(self.fc1(pooled))
        x = self.dropout1(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout2(x)

        # Output logits
        logits = self.fc_out(x)
        return logits



def train(model, train_loader, val_loader, optimizer, criterion, device, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        total_samples = 0

        progress_bar_train = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs} - Training')
        for batch in progress_bar_train:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            progress_bar_train.set_postfix({'Train Loss': total_loss / total_samples,
                                            'Train Accuracy': total_correct / total_samples})

        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {total_loss / total_samples:.4f}, "
              f"Train Accuracy: {total_correct / total_samples:.4f}, "
              f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")


def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    val_loss = total_loss / len(val_loader)
    val_accuracy = total_correct / total_samples

    return val_loss, val_accuracy



tokenizer = AutoTokenizer.from_pretrained('csebuetnlp/banglabert')
model = BertClassifier(AutoModel.from_pretrained('csebuetnlp/banglabert'), num_classes=5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()


# Train model
train(model, train_loader, val_loader, optimizer, criterion, device, epochs=15)


HateBert-CNN

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score
from tqdm import tqdm

class BertCNNClassifier(nn.Module):
    def __init__(self, bert_model, num_classes, num_filters, kernel_size, dropout_rate):
        super(BertCNNClassifier, self).__init__()
        self.bert = bert_model
        self.conv1 = nn.Conv1d(in_channels=self.bert.config.hidden_size,
                               out_channels=num_filters,
                               kernel_size=kernel_size,
                               padding=1)
        self.dropout1 = nn.Dropout(dropout_rate)
        self.conv2 = nn.Conv1d(in_channels=num_filters,
                               out_channels=num_filters,
                               kernel_size=kernel_size,
                               padding=1)
        self.global_max_pool = nn.AdaptiveMaxPool1d(1)
        self.fc1 = nn.Linear(num_filters, num_filters)
        self.dropout2 = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(num_filters, num_classes)

    def forward(self, input_ids, attention_mask):
        # BERT encoding
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        bert_output = outputs.last_hidden_state  # (batch_size, seq_len, hidden_size)

        # Permute to fit Conv1d layer input format (batch_size, hidden_size, seq_len)
        bert_output = bert_output.permute(0, 2, 1)

        # Conv1D layers
        conv1_output = torch.relu(self.conv1(bert_output))
        conv1_output = self.dropout1(conv1_output)
        conv2_output = torch.relu(self.conv2(conv1_output))

        # Global Max Pooling
        pooled_output = self.global_max_pool(conv2_output).squeeze(2)

        # Fully connected layers
        fc1_output = torch.relu(self.fc1(pooled_output))
        fc1_output = self.dropout2(fc1_output)
        logits = self.fc2(fc1_output)

        return logits

# Initialize the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('csebuetnlp/banglabert')
model = BertCNNClassifier(AutoModel.from_pretrained('csebuetnlp/banglabert'), 
                          num_classes=5, 
                          num_filters=512, 
                          kernel_size=5, 
                          dropout_rate=0.2)

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define optimizer and criterion
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# Define the learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=2, verbose=True)

def train(model, train_loader, val_loader, optimizer, criterion, scheduler, device, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        total_samples = 0

        progress_bar_train = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs} - Training')
        for batch in progress_bar_train:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            progress_bar_train.set_postfix({'Train Loss': total_loss / total_samples,
                                            'Train Accuracy': total_correct / total_samples})

        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {total_loss / total_samples:.4f}, "
              f"Train Accuracy: {total_correct / total_samples:.4f}, "
              f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

        # Step the scheduler with the validation accuracy
        scheduler.step(val_accuracy)

def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    val_loss = total_loss / len(val_loader)
    val_accuracy = total_correct / total_samples

    return val_loss, val_accuracy


# Train the model
train(model, train_loader, val_loader, optimizer, criterion, scheduler, device, epochs=15)



Model Test Phase

In [ ]:
# Define a function to load the model weights
def load_model(model, path, device):
    model.load_state_dict(torch.load(path))
    model.to(device)
    print(f"Model loaded from {path}")

# Example: Load the final model weights
load_model(model, os.path.join(save_path, "Multi_model_final.pth"), device)

In [ ]:
from sklearn.metrics import classification_report

def test(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_loss = total_loss / len(test_loader)
    test_accuracy = total_correct / total_samples

    return test_loss, test_accuracy, all_predictions, all_labels

# Test the model
test_loss, test_accuracy, all_predictions, all_labels = test(model, test_loader, criterion, device)

# Calculate classification report
target_names = ['PoHS', 'ReHS', 'MisoHS', 'SlaHS', 'XenHS'] 
print("Test Loss: {:.4f}, Test Accuracy: {:.4f}".format(test_loss, test_accuracy))
print(classification_report(all_labels, all_predictions, target_names=target_names))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Generate confusion matrix
cm = confusion_matrix(all_labels, all_predictions)

# Create the figure
plt.figure(figsize=(10, 7))

# Plot the heatmap
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names)

# Add labels and title
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')

# Save the figure
plt.savefig('Multi_confusion_matrix.png')

# Show the plot
plt.show()


In [ ]:
# Error analysis: Identify misclassified examples
misclassified_indices = np.where(np.array(all_labels) != np.array(all_predictions))[0]
print(f"Number of misclassified samples: {len(misclassified_indices)}")

# Display some misclassified examples
for i in misclassified_indices[:15]:  # Display first 5 misclassified examples
    input_ids = test_loader.dataset[i]["input_ids"]
    true_label = target_names[test_loader.dataset[i]["labels"]]
    predicted_label = target_names[all_predictions[i]]
    input_text = tokenizer.decode(input_ids, skip_special_tokens=True)
    print(f"Text: {input_text}")
    print(f"True Label: {true_label}, Predicted Label: {predicted_label}\n")

In [ ]:
#128, 5, 10, 0.1
import os

# Save the trained model weights
save_path = "./model_weights"
os.makedirs(save_path, exist_ok=True)
torch.save(model.state_dict(), os.path.join(save_path, "Multi_model_final_128.pth"))

print(f"Model weights saved to {os.path.join(save_path, 'Multi_model_final_128.pth')}")

In [ ]:
import os

# Save the trained model weights
save_path = "./model_weights"
os.makedirs(save_path, exist_ok=True)
torch.save(model.state_dict(), os.path.join(save_path, "Multi_model_final_512.pth"))

print(f"Model weights saved to {os.path.join(save_path, 'Multi_model_final_512.pth')}")

# Binary Hate Speech Classification 

In [ ]:
# import pandas as pd
# datapath = '/kaggle/input/binary-bd-hate-speech/Binary_BDHate.xlsx'
# df = pd.read_excel(datapath)
# df.head()

In [ ]:
import re

def cleaning_data(row):
    # Remove emojis
    text = re.sub(r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F700-\U0001F77F\U0001F780-\U0001F7FF\U0001F800-\U0001F8FF\U0001F900-\U0001F9FF\U0001FA00-\U0001FA6F\U0001FA70-\U0001FAFF\U00002702-\U000027B0\U000024C2-\U0001F251\U0001f926-\U0001f937\U0001F1E0-\U0001F1FF]', '', str(row))
    # Remove unnecessary punctuation and non-Bangla characters
    text = re.sub(r'[^\w\s\u0980-\u09FF]', '', text)
    return text

In [ ]:
df['Comments'] = df['Comments'].apply(cleaning_data)

In [ ]:
df.head()

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained('csebuetnlp/banglabert')

class HateSpeechDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        inputs = self.tokenizer(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_token_type_ids=False,
            return_attention_mask=True,
            return_tensors="pt",
        )

        return {
            "input_ids": inputs["input_ids"].flatten(),
            "attention_mask": inputs["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long),
        }

# Assuming you have a dataframe `df` with 'Comments' and 'Labels' columns
texts = df['Comments'].tolist()
labels = df['Label'].tolist()

# Split data into train, validation, and test sets
train_texts, test_texts, train_labels, test_labels = train_test_split(texts, labels, test_size=0.2, random_state=42, stratify=labels)
val_texts, test_texts, val_labels, test_labels = train_test_split(test_texts, test_labels, test_size=0.5, random_state=42, shuffle=False)

# Define training, validation, and test datasets
train_dataset = HateSpeechDataset(train_texts, train_labels, tokenizer, max_len=50)
val_dataset = HateSpeechDataset(val_texts, val_labels, tokenizer, max_len=50)
test_dataset = HateSpeechDataset(test_texts, test_labels, tokenizer, max_len=50)

# Define dataloaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))


BanglaBert-CNN

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score
from tqdm import tqdm

class BertCNNClassifier(nn.Module):
    def __init__(self, bert_model, num_classes, num_filters, kernel_size, dropout_rate):
        super(BertCNNClassifier, self).__init__()
        self.bert = bert_model
        self.conv1 = nn.Conv1d(in_channels=self.bert.config.hidden_size,
                               out_channels=num_filters,
                               kernel_size=kernel_size,
                               padding=1)
        self.dropout1 = nn.Dropout(dropout_rate)
        self.conv2 = nn.Conv1d(in_channels=num_filters,
                               out_channels=num_filters,
                               kernel_size=kernel_size,
                               padding=1)
        self.global_max_pool = nn.AdaptiveMaxPool1d(1)
        self.fc1 = nn.Linear(num_filters, num_filters)
        self.dropout2 = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(num_filters, num_classes)

    def forward(self, input_ids, attention_mask):
        # BERT encoding
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        bert_output = outputs.last_hidden_state  # (batch_size, seq_len, hidden_size)

        # Permute to fit Conv1d layer input format (batch_size, hidden_size, seq_len)
        bert_output = bert_output.permute(0, 2, 1)

        # Conv1D layers
        conv1_output = torch.relu(self.conv1(bert_output))
        conv1_output = self.dropout1(conv1_output)
        conv2_output = torch.relu(self.conv2(conv1_output))

        # Global Max Pooling
        pooled_output = self.global_max_pool(conv2_output).squeeze(2)

        # Fully connected layers
        fc1_output = torch.relu(self.fc1(pooled_output))
        fc1_output = self.dropout2(fc1_output)
        logits = self.fc2(fc1_output)

        return logits

# Initialize the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('csebuetnlp/banglabert')
model = BertCNNClassifier(AutoModel.from_pretrained('csebuetnlp/banglabert'), 
                          num_classes=2, 
                          num_filters=512, 
                          kernel_size=3, 
                          dropout_rate=0.2)

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define optimizer and criterion
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
criterion = nn.CrossEntropyLoss()

# Define the learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=1, verbose=True)

def train(model, train_loader, val_loader, optimizer, criterion, scheduler, device, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        total_samples = 0

        progress_bar_train = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs} - Training')
        for batch in progress_bar_train:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            progress_bar_train.set_postfix({'Train Loss': total_loss / total_samples,
                                            'Train Accuracy': total_correct / total_samples})

        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {total_loss / total_samples:.4f}, "
              f"Train Accuracy: {total_correct / total_samples:.4f}, "
              f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

        # Step the scheduler with the validation accuracy
        scheduler.step(val_accuracy)

def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    val_loss = total_loss / len(val_loader)
    val_accuracy = total_correct / total_samples

    return val_loss, val_accuracy

# Assume train_loader and val_loader are already defined
# Train the model
train(model, train_loader, val_loader, optimizer, criterion, scheduler, device, epochs=15)


HateBert-LSTM

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np

class BertLSTMClassifier(nn.Module):
    def __init__(self, bert_model, lstm_hidden_size, num_classes):
        super(BertLSTMClassifier, self).__init__()
        self.bert = bert_model
        self.lstm1 = nn.LSTM(input_size=self.bert.config.hidden_size,
                             hidden_size=lstm_hidden_size,
                             num_layers=1,
                             batch_first=True,
                             dropout=0.3,
                             bidirectional=False)
        self.dropout1 = nn.Dropout(0.3)
        self.lstm2 = nn.LSTM(input_size=lstm_hidden_size,
                             hidden_size=lstm_hidden_size,
                             num_layers=1,
                             batch_first=True,
                             dropout=0.3,
                             bidirectional=False)
        self.dropout2 = nn.Dropout(0.3)
        self.global_max_pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(lstm_hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        # BERT encoding
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        bert_output = outputs.last_hidden_state

        # First LSTM layer
        lstm_output1, _ = self.lstm1(bert_output)
        lstm_output1 = self.dropout1(lstm_output1)

        # Second LSTM layer
        lstm_output2, _ = self.lstm2(lstm_output1)
        lstm_output2, _ = self.dropout2(lstm_output2)
        # Global Max Pooling
        lstm_output2 = lstm_output2.permute(0, 2, 1)
        pooled_output = self.global_max_pool(lstm_output2).squeeze(2)

        # Classification layer
        logits = self.fc(pooled_output)

        return logits
def train(model, train_loader, val_loader, optimizer, criterion, scheduler, device, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        total_samples = 0

        progress_bar_train = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs} - Training')
        for batch in progress_bar_train:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            progress_bar_train.set_postfix({'Train Loss': total_loss / total_samples,
                                            'Train Accuracy': total_correct / total_samples})

        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {total_loss / total_samples:.4f}, "
              f"Train Accuracy: {total_correct / total_samples:.4f}, "
              f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

        # Step the scheduler after each epoch
        scheduler.step()

def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    val_loss = total_loss / len(val_loader)
    val_accuracy = total_correct / total_samples

    return val_loss, val_accuracy

def test(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_loss = total_loss / len(test_loader)
    test_accuracy = total_correct / total_samples

    return test_loss, test_accuracy, all_predictions, all_labels

# Initialize the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('csebuetnlp/banglabert')
model = BertLSTMClassifier(AutoModel.from_pretrained('csebuetnlp/banglabert'), lstm_hidden_size=512, num_classes=5)

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define optimizer and criterion
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# Define the learning rate scheduler
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.9)

# Train the model
train(model, train_loader, val_loader, optimizer, criterion, scheduler, device, epochs=15)

HateBert-MLP

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score
from tqdm import tqdm


class BertClassifier(nn.Module):
    def __init__(self, bert_model, num_classes):
        super(BertClassifier, self).__init__()
        self.bert = bert_model
        
        hidden_size = self.bert.config.hidden_size
        
        # Global Max Pooling
        self.global_max_pool = nn.AdaptiveMaxPool1d(1)
        
        # Multi-Layer Perceptron (MLP)
        self.fc1 = nn.Linear(hidden_size, 512)
        self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(512, 256)
        self.dropout2 = nn.Dropout(0.3)
        
        # Final classification layer
        self.fc_out = nn.Linear(256, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state  # shape: (batch_size, seq_len, hidden_size)

       
        pooled = self.global_max_pool(last_hidden_state.permute(0, 2, 1)).squeeze(-1)  # (batch_size, hidden_size)
    
        x = torch.relu(self.fc1(pooled))
        x = self.dropout1(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout2(x)

        # Output logits
        logits = self.fc_out(x)
        return logits



def train(model, train_loader, val_loader, optimizer, criterion, device, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        total_samples = 0

        progress_bar_train = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs} - Training')
        for batch in progress_bar_train:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            progress_bar_train.set_postfix({'Train Loss': total_loss / total_samples,
                                            'Train Accuracy': total_correct / total_samples})

        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {total_loss / total_samples:.4f}, "
              f"Train Accuracy: {total_correct / total_samples:.4f}, "
              f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")


def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    val_loss = total_loss / len(val_loader)
    val_accuracy = total_correct / total_samples

    return val_loss, val_accuracy



tokenizer = AutoTokenizer.from_pretrained('csebuetnlp/banglabert')
model = BertClassifier(AutoModel.from_pretrained('csebuetnlp/banglabert'), num_classes=5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()


# Train model
train(model, train_loader, val_loader, optimizer, criterion, device, epochs=15)

Model Test Phase

In [ ]:
# Test the model
test_loss, test_accuracy, all_predictions, all_labels = test(model, test_loader, criterion, device)

# Calculate classification report
target_names = ['NHS', 'HS']
print("Test Loss: {:.4f}, Test Accuracy: {:.4f}".format(test_loss, test_accuracy))
print(classification_report(all_labels, all_predictions, target_names=target_names))





In [ ]:
# Test the model
test_loss, test_accuracy, all_predictions, all_labels = test(model, test_loader, criterion, device)

# Calculate classification report
target_names = ['NHS', 'HS']
print("Test Loss: {:.4f}, Test Accuracy: {:.4f}".format(test_loss, test_accuracy))
print(classification_report(all_labels, all_predictions, target_names=target_names))


In [ ]:
# Generate confusion matrix
cm = confusion_matrix(all_labels, all_predictions)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.savefig('binary_confusion_matrix.png')
plt.show()

In [ ]:
# Error analysis: Identify misclassified examples
misclassified_indices = np.where(np.array(all_labels) != np.array(all_predictions))[0]
print(f"Number of misclassified samples: {len(misclassified_indices)}")

# Display some misclassified examples
for i in misclassified_indices[:5]:  # Display first 5 misclassified examples
    input_ids = test_loader.dataset[i]["input_ids"]
    true_label = target_names[test_loader.dataset[i]["labels"]]
    predicted_label = target_names[all_predictions[i]]
    input_text = tokenizer.decode(input_ids, skip_special_tokens=True)
    print(f"Text: {input_text}")
    print(f"True Label: {true_label}, Predicted Label: {predicted_label}\n")

In [ ]:
import os

# Save the trained model weights
save_path = "./model_weights"
os.makedirs(save_path, exist_ok=True)
torch.save(model.state_dict(), os.path.join(save_path, "model_final.pth"))

print(f"Model weights saved to {os.path.join(save_path, 'model_final.pth')}")


In [ ]:
save_path = "/kaggle/input/128-bn-weights/"
# Define a function to load the model weights
def load_model(model, path, device):
    model.load_state_dict(torch.load(path))
    model.to(device)
    print(f"Model loaded from {path}")

# Example: Load the final model weights
load_model(model, os.path.join(save_path, "model_final (3).pth"), device)

# **Other SOTA Model**

**For Binary**

In [ ]:
import pandas as pd
datapath = '/kaggle/input/binary-bd-hate-speech/Binary_BDHate.xlsx'
df = pd.read_excel(datapath)
df.head()

In [ ]:
import re

def cleaning_data(row):
    # Remove emojis
    text = re.sub(r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F700-\U0001F77F\U0001F780-\U0001F7FF\U0001F800-\U0001F8FF\U0001F900-\U0001F9FF\U0001FA00-\U0001FA6F\U0001FA70-\U0001FAFF\U00002702-\U000027B0\U000024C2-\U0001F251\U0001f926-\U0001f937\U0001F1E0-\U0001F1FF]', '', str(row))
    # Remove unnecessary punctuation and non-Bangla characters
    text = re.sub(r'[^\w\s\u0980-\u09FF]', '', text)
    return text


In [ ]:
df['Comments'] = df['Comments'].apply(cleaning_data)

In [ ]:
df.head()

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

# Initialize the tokenizer
# tokenizer = AutoTokenizer.from_pretrained('csebuetnlp/banglabert')
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-multilingual-cased')
# tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')

class HateSpeechDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        inputs = self.tokenizer(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_token_type_ids=False,
            return_attention_mask=True,
            return_tensors="pt",
        )

        return {
            "input_ids": inputs["input_ids"].flatten(),
            "attention_mask": inputs["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long),
        }

# Assuming you have a dataframe `df` with 'Comments' and 'Labels' columns
texts = df['Comments'].tolist()
labels = df['Label'].tolist()

# Split data into train, validation, and test sets
train_texts, test_texts, train_labels, test_labels = train_test_split(texts, labels, test_size=0.2, random_state=42, stratify=labels)
val_texts, test_texts, val_labels, test_labels = train_test_split(test_texts, test_labels, test_size=0.5, random_state=42, shuffle=False)

# Define training, validation, and test datasets
train_dataset = HateSpeechDataset(train_texts, train_labels, tokenizer, max_len=50)
val_dataset = HateSpeechDataset(val_texts, val_labels, tokenizer, max_len=50)
test_dataset = HateSpeechDataset(test_texts, test_labels, tokenizer, max_len=50)

# Define dataloaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8,shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8,shuffle=False)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))


Bangla-Bert 

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score
from tqdm import tqdm

class BertClassifier(nn.Module):
    def __init__(self, bert_model, num_classes):
        super(BertClassifier, self).__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.3)  # Dropout layer
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)  # Classification layer

    def forward(self, input_ids, attention_mask):
        # BERT encoding
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state  # Extract last hidden state
        
        # Use the [CLS] token representation (first token)
        cls_output = last_hidden_state[:, 0, :]
        
        # Dropout layer
        cls_output = self.dropout(cls_output)
        
        # Classification layer
        logits = self.fc(cls_output)
        
        return logits

def train(model, train_loader, val_loader, optimizer, criterion, device, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        total_samples = 0

        progress_bar_train = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs} - Training')
        for batch in progress_bar_train:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            progress_bar_train.set_postfix({'Train Loss': total_loss / total_samples,
                                            'Train Accuracy': total_correct / total_samples})

        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {total_loss / total_samples:.4f}, "
              f"Train Accuracy: {total_correct / total_samples:.4f}, "
              f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    val_loss = total_loss / len(val_loader)
    val_accuracy = total_correct / total_samples

    return val_loss, val_accuracy

# Initialize the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('csebuetnlp/banglabert')
model = BertClassifier(AutoModel.from_pretrained('csebuetnlp/banglabert'), num_classes=2)

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define optimizer and criterion
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()



# Train the model
train(model, train_loader, val_loader, optimizer, criterion, device, epochs=15)



In [ ]:
def test(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_loss = total_loss / len(test_loader)
    test_accuracy = total_correct / total_samples

    return test_loss, test_accuracy, all_predictions, all_labels


# Test the model
test_loss, test_accuracy, all_predictions, all_labels = test(model, test_loader, criterion, device)

# Calculate classification report
target_names = ['NHS', 'HS']
print("Test Loss: {:.4f}, Test Accuracy: {:.4f}".format(test_loss, test_accuracy))
print(classification_report(all_labels, all_predictions, target_names=target_names))

M-Bert

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score
from tqdm import tqdm

class BertClassifier(nn.Module):
    def __init__(self, bert_model, num_classes):
        super(BertClassifier, self).__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.3)  # Dropout layer
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)  # Classification layer

    def forward(self, input_ids, attention_mask):
        # BERT encoding
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state  # Extract last hidden state
        
        # Use the [CLS] token representation (first token)
        cls_output = last_hidden_state[:, 0, :]
        
        # Dropout layer
        cls_output = self.dropout(cls_output)
        
        # Classification layer
        logits = self.fc(cls_output)
        
        return logits

def train(model, train_loader, val_loader, optimizer, criterion, device, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        total_samples = 0

        progress_bar_train = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs} - Training')
        for batch in progress_bar_train:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            progress_bar_train.set_postfix({'Train Loss': total_loss / total_samples,
                                            'Train Accuracy': total_correct / total_samples})

        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {total_loss / total_samples:.4f}, "
              f"Train Accuracy: {total_correct / total_samples:.4f}, "
              f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    val_loss = total_loss / len(val_loader)
    val_accuracy = total_correct / total_samples

    return val_loss, val_accuracy

# Initialize the tokenizer and model for M-BERT
tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')
model = BertClassifier(AutoModel.from_pretrained('bert-base-multilingual-cased'), num_classes=2)

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define optimizer and criterion
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# # Dummy data loaders (replace with actual data loaders)
# train_loader = DataLoader(Dataset(), batch_size=32, shuffle=True)
# val_loader = DataLoader(Dataset(), batch_size=32)

# Train the model
train(model, train_loader, val_loader, optimizer, criterion, device, epochs=15)


In [ ]:
def test(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_loss = total_loss / len(test_loader)
    test_accuracy = total_correct / total_samples

    return test_loss, test_accuracy, all_predictions, all_labels


# Test the model
test_loss, test_accuracy, all_predictions, all_labels = test(model, test_loader, criterion, device)

# Calculate classification report
target_names = ['NHS', 'HS']
print("Test Loss: {:.4f}, Test Accuracy: {:.4f}".format(test_loss, test_accuracy))
print(classification_report(all_labels, all_predictions, target_names=target_names))

Distil-Bert

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score
from tqdm import tqdm

class DistilBertClassifier(nn.Module):
    def __init__(self, distilbert_model, num_classes):
        super(DistilBertClassifier, self).__init__()
        self.distilbert = distilbert_model
        self.dropout = nn.Dropout(0.3)  # Dropout layer
        self.fc = nn.Linear(self.distilbert.config.hidden_size, num_classes)  # Classification layer

    def forward(self, input_ids, attention_mask):
        # DistilBERT encoding
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs.last_hidden_state  # Extract last hidden state
        
        # Use the [CLS] token representation (first token)
        cls_output = hidden_state[:, 0, :]
        
        # Dropout layer
        cls_output = self.dropout(cls_output)
        
        # Classification layer
        logits = self.fc(cls_output)
        
        return logits

def train(model, train_loader, val_loader, optimizer, criterion, device, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        total_samples = 0

        progress_bar_train = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs} - Training')
        for batch in progress_bar_train:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            progress_bar_train.set_postfix({'Train Loss': total_loss / total_samples,
                                            'Train Accuracy': total_correct / total_samples})

        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {total_loss / total_samples:.4f}, "
              f"Train Accuracy: {total_correct / total_samples:.4f}, "
              f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    val_loss = total_loss / len(val_loader)
    val_accuracy = total_correct / total_samples

    return val_loss, val_accuracy

# Initialize the tokenizer and model for DistilBERT
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-multilingual-cased')
model = DistilBertClassifier(AutoModel.from_pretrained('distilbert-base-multilingual-cased'), num_classes=2)

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define optimizer and criterion
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()


# Train the model
train(model, train_loader, val_loader, optimizer, criterion, device, epochs=15)


In [ ]:
def test(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_loss = total_loss / len(test_loader)
    test_accuracy = total_correct / total_samples

    return test_loss, test_accuracy, all_predictions, all_labels


# Test the model
test_loss, test_accuracy, all_predictions, all_labels = test(model, test_loader, criterion, device)

# Calculate classification report
target_names = ['NHS', 'HS']
print("Test Loss: {:.4f}, Test Accuracy: {:.4f}".format(test_loss, test_accuracy))
print(classification_report(all_labels, all_predictions, target_names=target_names))

XLM-R

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score
from tqdm import tqdm

class BertClassifier(nn.Module):
    def __init__(self, bert_model, num_classes):
        super(BertClassifier, self).__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.3)  # Dropout layer
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)  # Classification layer

    def forward(self, input_ids, attention_mask):
        # BERT encoding
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state  # Extract last hidden state
        
        # Use the [CLS] token representation (first token)
        cls_output = last_hidden_state[:, 0, :]
        
        # Dropout layer
        cls_output = self.dropout(cls_output)
        
        # Classification layer
        logits = self.fc(cls_output)
        
        return logits

def train(model, train_loader, val_loader, optimizer, criterion, device, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        total_samples = 0

        progress_bar_train = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs} - Training')
        for batch in progress_bar_train:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            progress_bar_train.set_postfix({'Train Loss': total_loss / total_samples,
                                            'Train Accuracy': total_correct / total_samples})

        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {total_loss / total_samples:.4f}, "
              f"Train Accuracy: {total_correct / total_samples:.4f}, "
              f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    val_loss = total_loss / len(val_loader)
    val_accuracy = total_correct / total_samples

    return val_loss, val_accuracy

# Initialize the tokenizer and model for XLM-R
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = BertClassifier(AutoModel.from_pretrained('xlm-roberta-base'), num_classes=2)

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define optimizer and criterion
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# # Dummy data loaders (replace with actual data loaders)
# train_loader = DataLoader(Dataset(), batch_size=32, shuffle=True)
# val_loader = DataLoader(Dataset(), batch_size=32)

# Train the model
train(model, train_loader, val_loader, optimizer, criterion, device, epochs=15)


In [ ]:
def test(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_loss = total_loss / len(test_loader)
    test_accuracy = total_correct / total_samples

    return test_loss, test_accuracy, all_predictions, all_labels


# Test the model
test_loss, test_accuracy, all_predictions, all_labels = test(model, test_loader, criterion, device)

# Calculate classification report
target_names = ['NHS', 'HS']
print("Test Loss: {:.4f}, Test Accuracy: {:.4f}".format(test_loss, test_accuracy))
print(classification_report(all_labels, all_predictions, target_names=target_names))

# MultiClass

In [ ]:
import pandas as pd
datapath = '/kaggle/input/final-multi-hs/Final_multi_HS.xlsx'
df = pd.read_excel(datapath)
df.head()

In [ ]:
import re

def cleaning_data(row):
    # Remove emojis
    text = re.sub(r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F700-\U0001F77F\U0001F780-\U0001F7FF\U0001F800-\U0001F8FF\U0001F900-\U0001F9FF\U0001FA00-\U0001FA6F\U0001FA70-\U0001FAFF\U00002702-\U000027B0\U000024C2-\U0001F251\U0001f926-\U0001f937\U0001F1E0-\U0001F1FF]', '', str(row))
    # Remove unnecessary punctuation and non-Bangla characters
    text = re.sub(r'[^\w\s\u0980-\u09FF]', '', text)
    return text


In [ ]:
df['Comments'] = df['Comments'].apply(cleaning_data)

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

# Define the labels dictionary with a different name
label_dict = {
    'anti-political': 0,
    'anti-religious': 1,
    'misogynist': 2,
    'xenophobic': 4,
    'slander': 3
}

# Initialize the tokenizer
# tokenizer = AutoTokenizer.from_pretrained('csebuetnlp/banglabert')
# tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')
# tokenizer = AutoTokenizer.from_pretrained('distilbert-base-multilingual-cased')
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')

class HateSpeechDataset(Dataset):
    def __init__(self, texts, categories, tokenizer, max_len):
        self.texts = texts
        self.categories = categories
        self.tokenizer = tokenizer
        self.max_len = max_len

        # Encode categories manually using the label_dict dictionary
        self.labels = [label_dict[category] for category in categories]
        
        # Print labels with their actual names and encoded values
#         print("Labels with their actual names and encoded values:")
#         for category, encoded_value in zip(categories, self.labels):
#             print(f"{category}: {encoded_value}")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        inputs = self.tokenizer(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_token_type_ids=False,
            return_attention_mask=True,
            return_tensors="pt",
        )

        return {
            "input_ids": inputs["input_ids"].flatten(),
            "attention_mask": inputs["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long),
        }

# Assuming you have a dataframe `df` with 'Comments' and 'Category' columns
texts = df['Comments'].tolist()
categories = df['Category'].tolist()

# Split data into train, validation, and test sets
train_texts, test_texts, train_labels, test_labels = train_test_split(texts, categories, test_size=0.2, random_state=42, stratify=categories)
val_texts, test_texts, val_labels, test_labels = train_test_split(test_texts, test_labels, test_size=0.5, random_state=42, shuffle=False)

# Define training, validation, and test datasets
train_dataset = HateSpeechDataset(train_texts, train_labels, tokenizer, max_len=60)
val_dataset = HateSpeechDataset(val_texts, val_labels, tokenizer, max_len=60)
test_dataset = HateSpeechDataset(test_texts, test_labels, tokenizer, max_len=60)

# Define dataloaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))


Bangla-Bert

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score
from tqdm import tqdm

class BertClassifier(nn.Module):
    def __init__(self, bert_model, num_classes):
        super(BertClassifier, self).__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.3)  # Dropout layer
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)  # Classification layer

    def forward(self, input_ids, attention_mask):
        # BERT encoding
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state  # Extract last hidden state
        
        # Use the [CLS] token representation (first token)
        cls_output = last_hidden_state[:, 0, :]
        
        # Dropout layer
        cls_output = self.dropout(cls_output)
        
        # Classification layer
        logits = self.fc(cls_output)
        
        return logits

def train(model, train_loader, val_loader, optimizer, criterion, device, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        total_samples = 0

        progress_bar_train = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs} - Training')
        for batch in progress_bar_train:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            progress_bar_train.set_postfix({'Train Loss': total_loss / total_samples,
                                            'Train Accuracy': total_correct / total_samples})

        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {total_loss / total_samples:.4f}, "
              f"Train Accuracy: {total_correct / total_samples:.4f}, "
              f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    val_loss = total_loss / len(val_loader)
    val_accuracy = total_correct / total_samples

    return val_loss, val_accuracy

# Initialize the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('csebuetnlp/banglabert')
model = BertClassifier(AutoModel.from_pretrained('csebuetnlp/banglabert'), num_classes=5)

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define optimizer and criterion
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# # Dummy data loaders (replace with actual data loaders)
# train_loader = DataLoader(Dataset(), batch_size=32, shuffle=True)
# val_loader = DataLoader(Dataset(), batch_size=32)

# Train the model
train(model, train_loader, val_loader, optimizer, criterion, device, epochs=15)



In [ ]:
from sklearn.metrics import classification_report

def test(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_loss = total_loss / len(test_loader)
    test_accuracy = total_correct / total_samples

    return test_loss, test_accuracy, all_predictions, all_labels

# Test the model
test_loss, test_accuracy, all_predictions, all_labels = test(model, test_loader, criterion, device)

# Calculate classification report
target_names = ['PoHS', 'ReHS', 'MisoHS', 'SlaHS', 'XenHS'] 
print("Test Loss: {:.4f}, Test Accuracy: {:.4f}".format(test_loss, test_accuracy))
print(classification_report(all_labels, all_predictions, target_names=target_names))

M-Bert

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score
from tqdm import tqdm

class BertClassifier(nn.Module):
    def __init__(self, bert_model, num_classes):
        super(BertClassifier, self).__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.3)  # Dropout layer
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)  # Classification layer

    def forward(self, input_ids, attention_mask):
        # BERT encoding
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state  # Extract last hidden state
        
        # Use the [CLS] token representation (first token)
        cls_output = last_hidden_state[:, 0, :]
        
        # Dropout layer
        cls_output = self.dropout(cls_output)
        
        # Classification layer
        logits = self.fc(cls_output)
        
        return logits

def train(model, train_loader, val_loader, optimizer, criterion, device, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        total_samples = 0

        progress_bar_train = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs} - Training')
        for batch in progress_bar_train:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            progress_bar_train.set_postfix({'Train Loss': total_loss / total_samples,
                                            'Train Accuracy': total_correct / total_samples})

        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {total_loss / total_samples:.4f}, "
              f"Train Accuracy: {total_correct / total_samples:.4f}, "
              f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    val_loss = total_loss / len(val_loader)
    val_accuracy = total_correct / total_samples

    return val_loss, val_accuracy

# Initialize the tokenizer and model for M-BERT
tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')
model = BertClassifier(AutoModel.from_pretrained('bert-base-multilingual-cased'), num_classes=5)

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define optimizer and criterion
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# # Dummy data loaders (replace with actual data loaders)
# train_loader = DataLoader(Dataset(), batch_size=32, shuffle=True)
# val_loader = DataLoader(Dataset(), batch_size=32)

# Train the model
train(model, train_loader, val_loader, optimizer, criterion, device, epochs=15)


In [ ]:
from sklearn.metrics import classification_report

def test(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_loss = total_loss / len(test_loader)
    test_accuracy = total_correct / total_samples

    return test_loss, test_accuracy, all_predictions, all_labels

# Test the model
test_loss, test_accuracy, all_predictions, all_labels = test(model, test_loader, criterion, device)

# Calculate classification report
target_names = ['PoHS', 'ReHS', 'MisoHS', 'SlaHS', 'XenHS'] 
print("Test Loss: {:.4f}, Test Accuracy: {:.4f}".format(test_loss, test_accuracy))
print(classification_report(all_labels, all_predictions, target_names=target_names))

Distil-Bert

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score
from tqdm import tqdm

class DistilBertClassifier(nn.Module):
    def __init__(self, distilbert_model, num_classes):
        super(DistilBertClassifier, self).__init__()
        self.distilbert = distilbert_model
        self.dropout = nn.Dropout(0.3)  # Dropout layer
        self.fc = nn.Linear(self.distilbert.config.hidden_size, num_classes)  # Classification layer

    def forward(self, input_ids, attention_mask):
        # DistilBERT encoding
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs.last_hidden_state  # Extract last hidden state
        
        # Use the [CLS] token representation (first token)
        cls_output = hidden_state[:, 0, :]
        
        # Dropout layer
        cls_output = self.dropout(cls_output)
        
        # Classification layer
        logits = self.fc(cls_output)
        
        return logits

def train(model, train_loader, val_loader, optimizer, criterion, device, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        total_samples = 0

        progress_bar_train = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs} - Training')
        for batch in progress_bar_train:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            progress_bar_train.set_postfix({'Train Loss': total_loss / total_samples,
                                            'Train Accuracy': total_correct / total_samples})

        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {total_loss / total_samples:.4f}, "
              f"Train Accuracy: {total_correct / total_samples:.4f}, "
              f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    val_loss = total_loss / len(val_loader)
    val_accuracy = total_correct / total_samples

    return val_loss, val_accuracy

# Initialize the tokenizer and model for DistilBERT
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-multilingual-cased')
model = DistilBertClassifier(AutoModel.from_pretrained('distilbert-base-multilingual-cased'), num_classes=5)

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define optimizer and criterion
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# # Dummy data loaders (replace with actual data loaders)
# train_loader = DataLoader(Dataset(), batch_size=32, shuffle=True)
# val_loader = DataLoader(Dataset(), batch_size=32)

# Train the model
train(model, train_loader, val_loader, optimizer, criterion, device, epochs=15)


In [ ]:
from sklearn.metrics import classification_report

def test(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_loss = total_loss / len(test_loader)
    test_accuracy = total_correct / total_samples

    return test_loss, test_accuracy, all_predictions, all_labels

# Test the model
test_loss, test_accuracy, all_predictions, all_labels = test(model, test_loader, criterion, device)

# Calculate classification report
target_names = ['PoHS', 'ReHS', 'MisoHS', 'SlaHS', 'XenHS'] 
print("Test Loss: {:.4f}, Test Accuracy: {:.4f}".format(test_loss, test_accuracy))
print(classification_report(all_labels, all_predictions, target_names=target_names))

XLM-R

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score
from tqdm import tqdm

class BertClassifier(nn.Module):
    def __init__(self, bert_model, num_classes):
        super(BertClassifier, self).__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.3)  # Dropout layer
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)  # Classification layer

    def forward(self, input_ids, attention_mask):
        # BERT encoding
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state  # Extract last hidden state
        
        # Use the [CLS] token representation (first token)
        cls_output = last_hidden_state[:, 0, :]
        
        # Dropout layer
        cls_output = self.dropout(cls_output)
        
        # Classification layer
        logits = self.fc(cls_output)
        
        return logits

def train(model, train_loader, val_loader, optimizer, criterion, device, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        total_samples = 0

        progress_bar_train = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs} - Training')
        for batch in progress_bar_train:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            progress_bar_train.set_postfix({'Train Loss': total_loss / total_samples,
                                            'Train Accuracy': total_correct / total_samples})

        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {total_loss / total_samples:.4f}, "
              f"Train Accuracy: {total_correct / total_samples:.4f}, "
              f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    val_loss = total_loss / len(val_loader)
    val_accuracy = total_correct / total_samples

    return val_loss, val_accuracy

# Initialize the tokenizer and model for XLM-R
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = BertClassifier(AutoModel.from_pretrained('xlm-roberta-base'), num_classes=5)

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define optimizer and criterion
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# # Dummy data loaders (replace with actual data loaders)
# train_loader = DataLoader(Dataset(), batch_size=32, shuffle=True)
# val_loader = DataLoader(Dataset(), batch_size=32)

# Train the model
train(model, train_loader, val_loader, optimizer, criterion, device, epochs=15)


In [ ]:
from sklearn.metrics import classification_report

def test(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_loss = total_loss / len(test_loader)
    test_accuracy = total_correct / total_samples

    return test_loss, test_accuracy, all_predictions, all_labels

# Test the model
test_loss, test_accuracy, all_predictions, all_labels = test(model, test_loader, criterion, device)

# Calculate classification report
target_names = ['PoHS', 'ReHS', 'MisoHS', 'SlaHS', 'XenHS'] 
print("Test Loss: {:.4f}, Test Accuracy: {:.4f}".format(test_loss, test_accuracy))
print(classification_report(all_labels, all_predictions, target_names=target_names))